# 11 - Binance Testnet and Paper Trading Execution Layer

## Objetivo del notebook

Este notebook valida la primera capa de ejecución del criptobot en un entorno seguro:

- Conexión pública con Binance Spot Testnet
- Validación opcional de endpoints firmados con claves de testnet
- Validación opcional de órdenes mediante `order/test`, sin envío al matching engine
- Simulación local de órdenes mediante un paper broker reproducible
- Persistencia de decisiones y estados en logs JSONL reutilizables por el dashboard

## Alcance

El objetivo no es demostrar rentabilidad, sino comprobar que la arquitectura puede pasar de una señal de trading a una decisión ejecutable y trazable sin exponer capital real.

## Limitaciones

- Las órdenes reales de testnet quedan desactivadas por defecto.
- La política de señal incluida es un placeholder para probar conectividad, broker y logs.
- La integración con modelos supervisados o RL debe añadirse después de validar esta capa de ejecución.


In [1]:
# ============================================================
# Imports and project paths
# Se preparan rutas robustas para ejecutar el notebook desde la carpeta notebooks o desde la raíz del proyecto.
# ============================================================

from __future__ import annotations

import os
import sys
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path
from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv

from src.execution.binance_spot_testnet import BinanceSpotTestnetClient
from src.execution.execution_logger import JsonlExecutionLogger, read_jsonl_logs
from src.execution.paper_broker import PaperBroker, PaperBrokerConfig

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
SRC_PATH = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source path exists: {SRC_PATH.exists()}")


Project root: C:\Users\elwya\Documents\programacion\python\jupyter notebook\M10\TFM\Repository\cryptobot-tfm
Source path exists: True


In [2]:
# ============================================================
# Load local environment configuration
# Las credenciales se cargan desde .env para evitar hardcodear secretos en el notebook.
# ============================================================

load_dotenv(PROJECT_ROOT / ".env")

SYMBOL = os.getenv("TRADING_SYMBOL", "DOGEUSDT")
INTERVAL = os.getenv("TRADING_INTERVAL", "5m")
BASE_URL = os.getenv("BINANCE_SPOT_TESTNET_BASE_URL", "https://testnet.binance.vision")
ALLOW_LIVE_TESTNET_ORDERS = os.getenv("ALLOW_LIVE_TESTNET_ORDERS", "false").lower() == "true"

config_summary = pd.DataFrame([
    {"parameter": "SYMBOL", "value": SYMBOL},
    {"parameter": "INTERVAL", "value": INTERVAL},
    {"parameter": "BASE_URL", "value": BASE_URL},
    {"parameter": "ALLOW_LIVE_TESTNET_ORDERS", "value": ALLOW_LIVE_TESTNET_ORDERS},
    {"parameter": "HAS_TESTNET_API_KEY", "value": bool(os.getenv("BINANCE_TESTNET_API_KEY"))},
    {"parameter": "HAS_TESTNET_API_SECRET", "value": bool(os.getenv("BINANCE_TESTNET_API_SECRET"))},
])

config_summary

,parameter,value
0,SYMBOL,DOGEUSDT
1,INTERVAL,5m
2,BASE_URL,https://testnet.binance.vision
3,ALLOW_LIVE_TESTNET_ORDERS,False
4,HAS_TESTNET_API_KEY,True
5,HAS_TESTNET_API_SECRET,True


## Interpretación - Configuración local

Esta tabla debe confirmar que el notebook apunta a Spot Testnet y que las órdenes reales de testnet siguen bloqueadas salvo activación explícita. Si las claves aparecen como ausentes, se podrán probar endpoints públicos y paper trading, pero no endpoints firmados.

In [3]:
# ============================================================
# Initialize execution components
# Se importa la capa de ejecución modular y se crea un run_id para separar ejecuciones dentro del log JSONL.
# ============================================================

client = BinanceSpotTestnetClient.from_env()

LOG_PATH = PROJECT_ROOT / "results" / "execution_logs" / "paper_trading.jsonl"
logger = JsonlExecutionLogger(LOG_PATH)

NOTEBOOK_NAME = "11_testnet_paper_trading"
RUN_ID = f"{NOTEBOOK_NAME}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"
_EVENT_SEQUENCE = 0

def log_execution_event(event_type: str, payload: dict) -> dict:
    """Write one execution event with notebook-level run_id and event_id metadata."""
    global _EVENT_SEQUENCE
    _EVENT_SEQUENCE += 1
    enriched_payload = {
        "run_id": RUN_ID,
        "event_id": f"{RUN_ID}_{_EVENT_SEQUENCE:04d}",
        "event_sequence": _EVENT_SEQUENCE,
        "notebook": NOTEBOOK_NAME,
        **payload,
    }
    return logger.log_event(event_type, enriched_payload)

broker = PaperBroker(
    PaperBrokerConfig(
        symbol=SYMBOL,
        initial_quote_balance=Decimal(os.getenv("PAPER_INITIAL_QUOTE_BALANCE", "1000")),
        fee_bps=Decimal(os.getenv("PAPER_FEE_BPS", "10")),
        slippage_bps=Decimal(os.getenv("PAPER_SLIPPAGE_BPS", "5")),
        max_position_quote_pct=Decimal(os.getenv("PAPER_MAX_POSITION_QUOTE_PCT", "0.25")),
        min_quote_order_qty=Decimal(os.getenv("PAPER_MIN_QUOTE_ORDER_QTY", "5")),
    )
)

print(f"Execution log path: {LOG_PATH}")
print(f"Execution run id: {RUN_ID}")


Execution log path: C:\Users\elwya\Documents\programacion\python\jupyter notebook\M10\TFM\Repository\cryptobot-tfm\results\execution_logs\paper_trading.jsonl
Execution run id: 11_testnet_paper_trading_20260602T135921_81408ee5


In [4]:
# ============================================================
# Public connectivity smoke test
# Se validan conectividad REST, hora de servidor y precio actual sin necesidad de credenciales.
# ============================================================

connectivity_payload = {
    "ping": client.ping(),
    "server_time": client.server_time(),
    "ticker": client.ticker_price(SYMBOL),
}

connectivity_summary = pd.DataFrame([
    {"check": "ping", "result": "ok" if connectivity_payload["ping"] == {} else str(connectivity_payload["ping"])},
    {"check": "server_time", "result": connectivity_payload["server_time"].get("serverTime")},
    {"check": "ticker_price", "result": connectivity_payload["ticker"].get("price")},
])

log_execution_event("connectivity_check", {"symbol": SYMBOL, "payload": connectivity_payload})
connectivity_summary


,check,result
0,ping,ok
1,server_time,1780408760698
2,ticker_price,0.09839000


## Interpretación - Conectividad pública

La conexión pública con Binance Spot Testnet queda validada. El `ping` responde correctamente, la hora de servidor se recibe sin error y el ticker de `DOGEUSDT` devuelve precio (`0.09845000`) para el símbolo configurado.

Esta prueba confirma conectividad, endpoint base y símbolo, pero no evalúa todavía permisos de cuenta ni aceptación de órdenes firmadas.

In [5]:
# ============================================================
# Symbol filters and exchange rules
# Se extraen filtros del símbolo para conocer restricciones de cantidad, precio y nocional mínimo antes de simular órdenes.
# ============================================================

exchange_info = client.exchange_info(SYMBOL)
symbol_info = exchange_info["symbols"][0]
filters_df = pd.DataFrame(symbol_info["filters"])

symbol_summary = pd.DataFrame([
    {"field": "symbol", "value": symbol_info.get("symbol")},
    {"field": "status", "value": symbol_info.get("status")},
    {"field": "baseAsset", "value": symbol_info.get("baseAsset")},
    {"field": "quoteAsset", "value": symbol_info.get("quoteAsset")},
    {"field": "orderTypes", "value": ", ".join(symbol_info.get("orderTypes", []))},
])

log_execution_event("symbol_rules_loaded", {"symbol": SYMBOL, "filters": symbol_info["filters"]})

symbol_summary, filters_df


(        field                                              value
 0      symbol                                           DOGEUSDT
 1      status                                            TRADING
 2   baseAsset                                               DOGE
 3  quoteAsset                                               USDT
 4  orderTypes  LIMIT, LIMIT_MAKER, MARKET, STOP_LOSS, STOP_LO...,
                filterType    minPrice       maxPrice    tickSize      minQty  \
 0            PRICE_FILTER  0.00001000  1000.00000000  0.00001000         NaN   
 1                LOT_SIZE         NaN            NaN         NaN  1.00000000   
 2           ICEBERG_PARTS         NaN            NaN         NaN         NaN   
 3         MARKET_LOT_SIZE         NaN            NaN         NaN  0.00000000   
 4          TRAILING_DELTA         NaN            NaN         NaN         NaN   
 5   PERCENT_PRICE_BY_SIDE         NaN            NaN         NaN         NaN   
 6                NOTIONAL         N

## Interpretación - Reglas del símbolo

`DOGEUSDT` aparece como símbolo activo (`TRADING`) en Spot Testnet, con `DOGE` como activo base y `USDT` como activo cotizado. Los tipos de orden disponibles incluyen órdenes `LIMIT`, `MARKET` y variantes stop, por lo que el símbolo es válido para las pruebas de ejecución previstas.

Los filtros relevantes también son coherentes con una prueba controlada: `PRICE_FILTER` usa `tickSize = 0.00001000`, `LOT_SIZE` exige cantidades enteras de DOGE (`minQty = 1`, `stepSize = 1`) y el filtro `NOTIONAL` marca un mínimo de `1 USDT`. Con una prueba de `quoteOrderQty = 10`, la validación de una orden market queda por encima del nocional mínimo.

In [6]:
# ============================================================
# Optional signed account check
# Esta celda comprueba endpoints firmados solo si existen claves de Spot Testnet en el .env y convierte errores de autenticación en diagnóstico no fatal.
# ============================================================

HAS_KEYS = bool(os.getenv("BINANCE_TESTNET_API_KEY")) and bool(os.getenv("BINANCE_TESTNET_API_SECRET"))
HAS_SIGNED_ACCOUNT_ACCESS = False

if HAS_KEYS:
    try:
        account_payload = client.account()
        balances_df = pd.DataFrame(account_payload.get("balances", []))
        balances_df["free"] = pd.to_numeric(balances_df["free"], errors="coerce")
        balances_df["locked"] = pd.to_numeric(balances_df["locked"], errors="coerce")
        non_zero_balances_df = balances_df[(balances_df["free"] > 0) | (balances_df["locked"] > 0)].copy()
        HAS_SIGNED_ACCOUNT_ACCESS = True
        signed_account_status_df = pd.DataFrame([{
            "status": "ok",
            "detail": "Signed account endpoint is reachable with the configured Spot Testnet keys",
            "probable_cause": None,
            "next_step": "Continue with optional order/test validation if needed",
        }])
        log_execution_event("signed_account_check", {"symbol": SYMBOL, "non_zero_balances": non_zero_balances_df.to_dict("records")})
    except RuntimeError as exc:
        error_message = str(exc)
        probable_cause = "The configured key is not accepted by Spot Testnet"
        next_step = "Create a Spot Testnet API key at https://testnet.binance.vision, update .env and restart the kernel before rerunning."
        if "-2015" in error_message:
            probable_cause = "Invalid key, IP restriction or insufficient permissions"
        non_zero_balances_df = pd.DataFrame()
        signed_account_status_df = pd.DataFrame([{
            "status": "failed",
            "detail": error_message,
            "probable_cause": probable_cause,
            "next_step": next_step,
        }])
        log_execution_event("signed_account_check_failed", {"symbol": SYMBOL, "error": error_message})
else:
    non_zero_balances_df = pd.DataFrame()
    signed_account_status_df = pd.DataFrame([{
        "status": "skipped",
        "detail": "Missing Spot Testnet API key or secret",
        "probable_cause": "BINANCE_TESTNET_API_KEY or BINANCE_TESTNET_API_SECRET is not defined in .env",
        "next_step": "Create a Spot Testnet API key, update .env and restart the kernel before rerunning",
    }])
    log_execution_event("signed_account_check_skipped", {"symbol": SYMBOL, "reason": "Missing Spot Testnet API key or secret"})

display(signed_account_status_df)
non_zero_balances_df.head(20)


,status,detail,probable_cause,next_step
0,ok,Signed account endpoint is reachable with the ...,None,Continue with optional order/test validation i...


,asset,free,locked
0,这是测试币,10000.0,0.0
1,456,10000.0,0.0
2,BNB,1.0,0.0
3,BTC,1.0,0.0
4,USDT,10000.0,0.0
5,ETH,1.0,0.0
6,LTC,8.0,0.0
7,TRX,1450.0,0.0
8,XRP,351.0,0.0
9,GAS,300.0,0.0


## Interpretación - Cuenta testnet

La consulta firmada de cuenta devuelve `status = ok`, así que las claves configuradas en `.env` pertenecen a Spot Testnet y son aceptadas por Binance para endpoints firmados.

La tabla de balances confirma además que la cuenta testnet tiene fondos virtuales disponibles, incluyendo `10000 USDT`, por lo que la parte de autenticación queda desbloqueada. No se observan claves ni secrets expuestos en la salida del notebook.

In [7]:
# ============================================================
# Forced paper buy and sell smoke test
# Se fuerza una compra y una venta paper para validar cambios de balance, posición, fees y logs sin enviar órdenes a Binance.
# ============================================================

RUN_FORCED_PAPER_BUY_SELL = True
FORCED_SMOKE_TEST_QUOTE_QTY = Decimal("10")

forced_smoke_test_price = Decimal(str(connectivity_payload["ticker"]["price"]))
forced_smoke_test_timestamp = datetime.now(timezone.utc).isoformat()
forced_initial_equity = float(broker.cash_quote + broker.position_base * forced_smoke_test_price)

if RUN_FORCED_PAPER_BUY_SELL:
    forced_buy_result = broker.execute_signal(
        signal="BUY",
        price=forced_smoke_test_price,
        quote_amount=FORCED_SMOKE_TEST_QUOTE_QTY,
        timestamp=forced_smoke_test_timestamp,
        reason="Forced BUY smoke test for paper broker validation",
    )
    forced_buy_log_event = log_execution_event("forced_paper_buy", forced_buy_result)

    forced_sell_quantity = Decimal(str(forced_buy_result.get("quantity_base", "0")))
    forced_sell_result = broker.execute_signal(
        signal="SELL",
        price=forced_smoke_test_price,
        quantity_base=forced_sell_quantity,
        timestamp=datetime.now(timezone.utc).isoformat(),
        reason="Forced SELL smoke test for paper broker validation",
    )
    forced_sell_log_event = log_execution_event("forced_paper_sell", forced_sell_result)

    forced_buy_display = {**forced_buy_result, "run_id": RUN_ID, "event_id": forced_buy_log_event["event_id"]}
    forced_sell_display = {**forced_sell_result, "run_id": RUN_ID, "event_id": forced_sell_log_event["event_id"]}
    forced_smoke_results_df = pd.DataFrame([forced_buy_display, forced_sell_display])

    forced_final_equity = float(broker.cash_quote + broker.position_base * forced_smoke_test_price)
    forced_smoke_summary_df = pd.DataFrame([{
        "run_id": RUN_ID,
        "status": "executed",
        "symbol": SYMBOL,
        "forced_quote_qty": float(FORCED_SMOKE_TEST_QUOTE_QTY),
        "test_price": float(forced_smoke_test_price),
        "initial_equity": forced_initial_equity,
        "final_equity_after_roundtrip": forced_final_equity,
        "roundtrip_return_pct": (forced_final_equity / forced_initial_equity - 1) * 100 if forced_initial_equity else None,
        "buy_event_id": forced_buy_log_event["event_id"],
        "sell_event_id": forced_sell_log_event["event_id"],
    }])
else:
    forced_smoke_results_df = pd.DataFrame()
    forced_smoke_summary_df = pd.DataFrame([{
        "run_id": RUN_ID,
        "status": "disabled",
        "symbol": SYMBOL,
        "reason": "Set RUN_FORCED_PAPER_BUY_SELL=True to validate paper BUY/SELL state transitions",
    }])
    log_execution_event("forced_paper_buy_sell_skipped", {"symbol": SYMBOL, "reason": "Forced paper smoke test disabled"})

display(forced_smoke_summary_df)
forced_smoke_results_df


,run_id,status,symbol,forced_quote_qty,test_price,initial_equity,final_equity_after_roundtrip,roundtrip_return_pct,buy_event_id,sell_event_id
0,11_testnet_paper_trading_20260602T135921_81408ee5,executed,DOGEUSDT,10.0,0.09839,1000.0,999.970035,-0.002997,11_testnet_paper_trading_20260602T135921_81408...,11_testnet_paper_trading_20260602T135921_81408...


,timestamp,symbol,side,mark_price,execution_price,quantity_base,gross_quote,fee_quote,cash_quote_after,position_base_after,equity_quote_after,reason,mode,run_id,event_id
0,2026-06-02T13:59:22.909951+00:00,DOGEUSDT,BUY,0.09839,0.098439,101.483967,10.000000,0.01000,990.000000,101.483967,999.985007,Forced BUY smoke test for paper broker validation,paper,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...
1,2026-06-02T13:59:22.916950+00:00,DOGEUSDT,SELL,0.09839,0.098341,101.483967,9.980015,0.00998,999.970035,0.000000,999.970035,Forced SELL smoke test for paper broker valida...,paper,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...


## Interpretación - Smoke test paper forzado

Esta prueba fuerza una entrada `BUY` y una salida `SELL` consecutivas usando el broker local. No se envía ninguna orden a Binance: el objetivo es comprobar que la cartera paper cambia de estado, aplica `fees`, incorpora `slippage`, actualiza equity y registra ambos eventos en el JSONL.

El retorno del roundtrip debería ser ligeramente negativo aunque el precio usado para comprar y vender sea el mismo. Esa pérdida no es un fallo de estrategia: es el coste esperado de cruzar compra y venta con comisiones y slippage. Si aparecen `forced_paper_buy` y `forced_paper_sell` con `run_id` y `event_id`, la capa de ejecución simulada queda validada para transiciones reales de balance.


In [8]:
# ============================================================
# Optional signed order validation through order/test
# Esta validación comprueba formato, firma y filtros de una orden sin enviarla al matching engine.
# ============================================================

RUN_SIGNED_TEST_ORDER = False
TEST_ORDER_QUOTE_QTY = Decimal("10")

if RUN_SIGNED_TEST_ORDER and HAS_KEYS and HAS_SIGNED_ACCOUNT_ACCESS:
    try:
        test_order_payload = client.test_order(
            symbol=SYMBOL,
            side="BUY",
            order_type="MARKET",
            quote_order_qty=TEST_ORDER_QUOTE_QTY,
        )
        test_order_result = pd.DataFrame([{
            "status": "validated",
            "detail": "order/test accepted the signed request",
            "payload": test_order_payload,
        }])
        log_execution_event("signed_test_order", {"symbol": SYMBOL, "side": "BUY", "quote_order_qty": float(TEST_ORDER_QUOTE_QTY)})
    except RuntimeError as exc:
        error_message = str(exc)
        test_order_result = pd.DataFrame([{
            "status": "failed",
            "detail": error_message,
            "probable_cause": "The signed request was rejected by Binance; check Spot Testnet key permissions, symbol filters and order parameters",
        }])
        log_execution_event("signed_test_order_failed", {"symbol": SYMBOL, "error": error_message})
elif RUN_SIGNED_TEST_ORDER and not HAS_KEYS:
    test_order_result = pd.DataFrame([{
        "status": "skipped",
        "reason": "Missing Spot Testnet API key or secret",
    }])
    log_execution_event("signed_test_order_skipped", {"symbol": SYMBOL, "reason": "Missing Spot Testnet API key or secret"})
elif RUN_SIGNED_TEST_ORDER and not HAS_SIGNED_ACCOUNT_ACCESS:
    test_order_result = pd.DataFrame([{
        "status": "skipped",
        "reason": "Signed account check did not pass; fix Spot Testnet credentials before validating orders",
    }])
    log_execution_event("signed_test_order_skipped", {"symbol": SYMBOL, "reason": "Signed account check did not pass"})
else:
    test_order_result = pd.DataFrame([{
        "status": "disabled",
        "reason": "Set RUN_SIGNED_TEST_ORDER=True after checking signed account access and symbol filters",
    }])
    log_execution_event("signed_test_order_disabled", {"symbol": SYMBOL, "reason": "RUN_SIGNED_TEST_ORDER=False"})

test_order_result


,status,reason
0,disabled,Set RUN_SIGNED_TEST_ORDER=True after checking ...


## Interpretación - Validación de orden

La validación mediante `order/test` sigue desactivada porque `RUN_SIGNED_TEST_ORDER = False`. Esto es correcto como valor seguro por defecto: el notebook ya ha probado la cuenta firmada y el broker paper sin colocar órdenes en Binance.

El smoke test forzado de `BUY`/`SELL` valida la mecánica local de cartera. `order/test` queda como comprobación adicional de firma, permisos y parámetros de orden contra Spot Testnet, sin ejecución real en el matching engine.


In [9]:
# ============================================================
# Recent market data snapshot
# Se descargan velas recientes para alimentar una política de señal mínima y comprobar el flujo end-to-end.
# ============================================================

klines = client.klines(SYMBOL, interval=INTERVAL, limit=50)
columns = [
    "open_time", "open", "high", "low", "close", "volume", "close_time", "quote_asset_volume",
    "number_of_trades", "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore",
]
market_df = pd.DataFrame(klines, columns=columns)

numeric_cols = ["open", "high", "low", "close", "volume", "quote_asset_volume"]
for col in numeric_cols:
    market_df[col] = pd.to_numeric(market_df[col], errors="coerce")

market_df["open_time"] = pd.to_datetime(market_df["open_time"], unit="ms")
market_df["close_time"] = pd.to_datetime(market_df["close_time"], unit="ms")
market_df["return_prev_1"] = market_df["close"].pct_change()
market_df["ema_10"] = market_df["close"].ewm(span=10, adjust=False).mean()
market_df["ema_20"] = market_df["close"].ewm(span=20, adjust=False).mean()

market_df.tail(5)

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore,return_prev_1,ema_10,ema_20
45,2026-06-02 13:35:00,0.09868,0.09874,0.09853,0.09860,262046.0,2026-06-02 13:39:59.999,25843.99995,34,212344.00000000,20941.14273000,0,-0.000811,0.098748,0.098767
46,2026-06-02 13:40:00,0.09860,0.09876,0.09852,0.09876,173692.0,2026-06-02 13:44:59.999,17132.85122,19,114132.00000000,11257.70392000,0,0.001623,0.098750,0.098767
47,2026-06-02 13:45:00,0.09881,0.09882,0.09856,0.09856,464633.0,2026-06-02 13:49:59.999,45854.47937,25,399543.00000000,39427.27019000,0,-0.002025,0.098716,0.098747
48,2026-06-02 13:50:00,0.09860,0.09864,0.09847,0.09855,339525.0,2026-06-02 13:54:59.999,33462.28879,26,181565.00000000,17894.49319000,0,-0.000101,0.098686,0.098728
49,2026-06-02 13:55:00,0.09849,0.09852,0.09839,0.09839,251351.0,2026-06-02 13:59:59.999,24744.28798,19,190251.00000000,18729.20338000,0,-0.001624,0.098632,0.098696


## Interpretación - Snapshot de mercado

La descarga de velas recientes funciona y produce un DataFrame operativo con OHLCV, volumen, número de trades, `return_prev_1`, `ema_10` y `ema_20`.

En el último tramo mostrado, el precio permanece alrededor de `0.09845` y la `ema_10` queda por debajo de la `ema_20`, lo que encaja con una señal conservadora de no entrada en la política placeholder. Esta sección valida ingesta y transformación mínima en tiempo casi real, no sustituye al pipeline histórico de features.

In [10]:
# ============================================================
# Placeholder signal policy
# Esta política mínima se usa solo para probar el funcionamiento de ejecución/logs, el modelo real debe conectarse después.
# ============================================================

latest_row = market_df.iloc[-1]
previous_row = market_df.iloc[-2]
latest_price = Decimal(str(latest_row["close"]))

if latest_row["ema_10"] > latest_row["ema_20"] and previous_row["ema_10"] <= previous_row["ema_20"]:
    signal = "BUY"
elif latest_row["ema_10"] < latest_row["ema_20"] and previous_row["ema_10"] >= previous_row["ema_20"]:
    signal = "SELL"
else:
    signal = "HOLD"

signal_payload = {
    "timestamp": latest_row["close_time"].isoformat(),
    "symbol": SYMBOL,
    "close": float(latest_price),
    "ema_10": float(latest_row["ema_10"]),
    "ema_20": float(latest_row["ema_20"]),
    "signal": signal,
    "reason": "EMA crossover smoke-test policy",
}

signal_log_event = log_execution_event("signal_generated", signal_payload)
signal_display = {**signal_payload, "run_id": RUN_ID, "event_id": signal_log_event["event_id"]}

pd.DataFrame([signal_display])


,timestamp,symbol,close,ema_10,ema_20,signal,reason,run_id,event_id
0,2026-06-02T13:59:59.999000,DOGEUSDT,0.09839,0.098632,0.098696,HOLD,EMA crossover smoke-test policy,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...


## Interpretación - Señal placeholder

La política de prueba genera `HOLD`. La decisión es coherente con las medias calculadas: `ema_10` (`0.098701`) queda por debajo de `ema_20` (`0.098830`) y no aparece un cruce alcista o bajista en la última vela.

Esta señal debe interpretarse solo como prueba de cableado. El resultado útil aquí es que una decisión discreta (`BUY`, `SELL` o `HOLD`) queda serializada con timestamp, símbolo, precio y razón operativa para ser enviada al broker simulado y al sistema de logs.

In [11]:
# ============================================================
# Paper execution
# Se ejecuta la señal contra el broker local con fees y slippage sin enviar ninguna orden a Binance.
# ============================================================

execution_result = broker.execute_signal(
    signal=signal,
    price=latest_price,
    quote_amount=Decimal("10"),
    timestamp=signal_payload["timestamp"],
    reason=signal_payload["reason"],
)

paper_execution_log_event = log_execution_event("paper_execution", execution_result)
execution_display = {**execution_result, "run_id": RUN_ID, "event_id": paper_execution_log_event["event_id"]}

pd.DataFrame([execution_display])


,timestamp,symbol,side,mark_price,execution_price,quantity_base,gross_quote,fee_quote,cash_quote_after,position_base_after,equity_quote_after,reason,mode,run_id,event_id
0,2026-06-02T13:59:59.999000,DOGEUSDT,HOLD,0.09839,None,0.0,0.0,0.0,999.970035,0.0,999.970035,EMA crossover smoke-test policy,paper,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...


## Interpretación - Ejecución paper

Esta celda ejecuta la señal placeholder contra el broker local después del smoke test forzado. Si la señal vuelve a ser `HOLD`, la cartera no debería abrir una nueva posición y el resultado se limitará a registrar un nuevo punto de equity.

La validación real de cambios de balance ya queda cubierta por la celda forzada de `BUY`/`SELL`. Por tanto, esta parte se interpreta como prueba de integración entre señal discreta, broker paper y logging, no como evaluación de rentabilidad de una estrategia.


In [12]:
# ============================================================
# Execution logs summary
# Se leen los logs JSONL y se separa la ejecución actual mediante run_id para evitar mezclar runs acumulados.
# ============================================================

logs_df = read_jsonl_logs(LOG_PATH)

if not logs_df.empty and "run_id" in logs_df.columns:
    current_run_logs_df = logs_df[logs_df["run_id"] == RUN_ID].copy()
else:
    current_run_logs_df = pd.DataFrame()

if not current_run_logs_df.empty:
    current_run_logs_summary_df = (
        current_run_logs_df
        .groupby(["run_id", "event_type"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["run_id", "event_type"])
    )
else:
    current_run_logs_summary_df = pd.DataFrame(columns=["run_id", "event_type", "count"])

if not logs_df.empty:
    all_logs_summary_df = (
        logs_df
        .groupby("event_type", dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("event_type")
    )
else:
    all_logs_summary_df = pd.DataFrame(columns=["event_type", "count"])

display(current_run_logs_summary_df)
display(all_logs_summary_df)
current_run_logs_df.tail(20)


,run_id,event_type,count
0,11_testnet_paper_trading_20260602T135921_81408ee5,connectivity_check,1
1,11_testnet_paper_trading_20260602T135921_81408ee5,forced_paper_buy,1
2,11_testnet_paper_trading_20260602T135921_81408ee5,forced_paper_sell,1
3,11_testnet_paper_trading_20260602T135921_81408ee5,paper_execution,1
4,11_testnet_paper_trading_20260602T135921_81408ee5,signal_generated,1
5,11_testnet_paper_trading_20260602T135921_81408ee5,signed_account_check,1
6,11_testnet_paper_trading_20260602T135921_81408ee5,signed_test_order_disabled,1
7,11_testnet_paper_trading_20260602T135921_81408ee5,symbol_rules_loaded,1


,event_type,count
0,connectivity_check,6
1,forced_paper_buy,1
2,forced_paper_sell,1
3,paper_execution,3
4,signal_generated,3
5,signed_account_check,3
6,signed_test_order_disabled,1
7,symbol_rules_loaded,6


,logged_at,event_type,symbol,payload,filters,non_zero_balances,timestamp,close,ema_10,ema_20,...,gross_quote,fee_quote,cash_quote_after,position_base_after,equity_quote_after,mode,run_id,event_id,event_sequence,notebook
16,2026-06-02T13:59:22.025182+00:00,connectivity_check,DOGEUSDT,"{'ping': {}, 'server_time': {'serverTime': 178...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,1.0,11_testnet_paper_trading
17,2026-06-02T13:59:22.298896+00:00,symbol_rules_loaded,DOGEUSDT,NaN,"[{'filterType': 'PRICE_FILTER', 'minPrice': '0...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,2.0,11_testnet_paper_trading
18,2026-06-02T13:59:22.885748+00:00,signed_account_check,DOGEUSDT,NaN,NaN,"[{'asset': '这是测试币', 'free': 10000.0, 'locked':...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,3.0,11_testnet_paper_trading
19,2026-06-02T13:59:22.915959+00:00,forced_paper_buy,DOGEUSDT,NaN,NaN,NaN,2026-06-02T13:59:22.909951+00:00,NaN,NaN,NaN,...,10.000000,0.01000,990.000000,101.483967,999.985007,paper,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,4.0,11_testnet_paper_trading
20,2026-06-02T13:59:22.916950+00:00,forced_paper_sell,DOGEUSDT,NaN,NaN,NaN,2026-06-02T13:59:22.916950+00:00,NaN,NaN,NaN,...,9.980015,0.00998,999.970035,0.000000,999.970035,paper,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,5.0,11_testnet_paper_trading
21,2026-06-02T13:59:22.957061+00:00,signed_test_order_disabled,DOGEUSDT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,6.0,11_testnet_paper_trading
22,2026-06-02T13:59:23.287800+00:00,signal_generated,DOGEUSDT,NaN,NaN,NaN,2026-06-02T13:59:59.999000,0.09839,0.098632,0.098696,...,NaN,NaN,NaN,NaN,NaN,NaN,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,7.0,11_testnet_paper_trading
23,2026-06-02T13:59:23.315358+00:00,paper_execution,DOGEUSDT,NaN,NaN,NaN,2026-06-02T13:59:59.999000,NaN,NaN,NaN,...,0.000000,0.00000,999.970035,0.000000,999.970035,paper,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading_20260602T135921_81408...,8.0,11_testnet_paper_trading


## Interpretación - Logs

Los logs JSONL incorporan `run_id`, `event_id`, `event_sequence` y `notebook`, lo que permite separar una ejecución concreta del histórico append-only. Esto resuelve el problema de que los conteos globales pueden seguir acumulando pruebas, pero el resumen por `RUN_ID` muestra únicamente los eventos del run actual.

Para este notebook deberían aparecer, como mínimo, eventos de conectividad, reglas del símbolo, cuenta firmada, smoke test paper forzado, validación `order/test` desactivada, señal placeholder y ejecución paper. Esta estructura ya es suficiente para que el dashboard filtre por ejecución y reconstruya una traza ordenada del bot.


In [13]:
# ============================================================
# Paper broker equity summary
# Se genera un resumen numérico mínimo para comparar estado inicial y estado final tras la ejecución simulada.
# ============================================================

equity_df = broker.equity_dataframe()
trades_df = broker.trades_dataframe()

if not equity_df.empty:
    initial_equity = float(broker.config.initial_quote_balance)
    final_equity = equity_df["equity_quote"].iloc[-1]
    return_pct = (final_equity / initial_equity - 1) * 100
    paper_summary_df = pd.DataFrame([
        {
            "run_id": RUN_ID,
            "notebook": NOTEBOOK_NAME,
            "strategy": "Execution-layer smoke test with forced paper roundtrip",
            "symbol": SYMBOL,
            "initial_balance": initial_equity,
            "final_equity": final_equity,
            "return_pct": return_pct,
            "number_of_trades": len(trades_df),
            "forced_paper_buy_sell": RUN_FORCED_PAPER_BUY_SELL,
            "fees_bps": float(broker.config.fee_bps),
            "slippage_bps": float(broker.config.slippage_bps),
            "limitations": "Smoke test only; includes forced paper BUY/SELL and no trained model is connected yet",
        }
    ])
else:
    paper_summary_df = pd.DataFrame()

SUMMARY_PATH = PROJECT_ROOT / "results" / "11_testnet_paper_trading_summary.csv"
if not paper_summary_df.empty:
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    paper_summary_df.to_csv(SUMMARY_PATH, index=False)

paper_summary_df


,run_id,notebook,strategy,symbol,initial_balance,final_equity,return_pct,number_of_trades,forced_paper_buy_sell,fees_bps,slippage_bps,limitations
0,11_testnet_paper_trading_20260602T135921_81408ee5,11_testnet_paper_trading,Execution-layer smoke test with forced paper r...,DOGEUSDT,1000.0,999.970035,-0.002997,2,True,10.0,5.0,Smoke test only; includes forced paper BUY/SEL...


## Interpretación - Paper Broker Summary

El summary final refleja el estado del broker después del roundtrip forzado y de la señal placeholder. `number_of_trades` debería ser mayor que cero porque el notebook ejecuta una compra y una venta paper para validar la mecánica de cartera.

Si `return_pct` queda ligeramente por debajo de cero, es el comportamiento esperado del smoke test: se compra y se vende al mismo precio de referencia, pero el broker aplica fees y slippage. Esta tabla no evalúa una estrategia rentable; documenta que la capa de ejecución simulada modifica balances, registra trades y persiste un resumen comparable en `../results/`.


# Conclusiones

El notebook queda preparado como validación completa de la capa inicial de Testnet + paper trading.

La conexión pública con Binance Spot Testnet se valida mediante `ping`, hora de servidor y precio para `DOGEUSDT`. El símbolo está activo (`TRADING`) y sus filtros principales son compatibles con pruebas controladas de órdenes market con nocional bajo.

La autenticación firmada también queda validada mediante `/api/v3/account`, siempre que las claves configuradas en `.env` pertenezcan a Spot Testnet. El `.env` permanece fuera de Git, por lo que la gestión básica de secretos queda razonablemente cubierta para el alcance académico del TFM.

Se añade un smoke test paper forzado con `BUY` y `SELL`. Esta prueba valida cambios reales de cash, posición, fees, slippage, equity y registro de trades sin enviar órdenes a Binance. El resultado esperado del roundtrip es ligeramente negativo por costes, no por fallo de la lógica.

`order/test` permanece desactivado por defecto con `RUN_SIGNED_TEST_ORDER = False`. Puede activarse una vez para validar firma, permisos y parámetros contra Spot Testnet sin colocar órdenes reales en el matching engine.

Los logs JSONL incorporan `run_id`, `event_id` y `event_sequence`, de modo que el dashboard podrá filtrar una ejecución concreta y reconstruir la secuencia de eventos sin confundirse con ejecuciones anteriores acumuladas.

Próximos pasos justificados:

- Ejecutar una vez `RUN_SIGNED_TEST_ORDER = True` si se quiere validar `order/test`
- Conectar una señal procedente del baseline ML o del agente RL
- Mantener la ejecución en modo paper antes de permitir órdenes reales de testnet
- Hacer que el dashboard lea `paper_trading.jsonl` filtrando por `run_id`
